# Part 1 - XAI on Tabular data

This notebook will take you through a tutorial on Explainable AI. The setup is as follows:

1. Create a Machine Learning model for the UCI Breastcancer dataset.
2. Implement (yourself!) Individual Conditional Expectation (ICE) plots, and observe how a variable of your choice influences the output.
3. Use SKLearn to create Partial Depence Plots (PDP) for the same variable. Are the PDP plots and ICE plots different? What do they say about your model?
4. Use the SHAP library to rank variable importance for all variables. Does this give you different insights into your model?
5. Based on what you learned here, how would you evaluate an ML model for such a task?

After completing Part 1, take some time to investigate your results and interpret them. We will have a group discussion about it, and some groups will be asked to present their findings.

Throughout the notebook, I recommend disabling Gemini / LLM support. The goal is to learn, and Gemini can get in the way of that.

## An ML model for Breast Cancer Classification

The task that we'll consider is classifying breast tumors as either Benign (not-cancer) or Malignant (uncontrolled growth, cancer), based on features of extracted tissue.

We will use this dataset: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

It is important for such a model to behave reasonably. It's not exactly clear what should be considered reasonable, but a good start will be to look at which features it uses, and how its response changes when the features change. We can then match this with prior knowledge or discuss with an oncologist.


Task 1:
- Get the dataset. It is easiest to get it through SKLearn, https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html. It may be nice to get this as Pandas Dataframes.
- Train & validate an SKLearn ML model. I'd recommend picking a non-linear model that gives class probabilities so you can see something interesting. For an SVM set `probability=True`. Avoid Neural Networks, since they are computationally expensive.
- Don't forget to properly normalise your data!
- Check that your model has a reasonable validation accuracy. Your accuracy should be somewhere >90%, but < 99%.

## Individual Condition Expectation Plots to analyse a Feature

Now that you have a trained model, it's time to observe the ICE of a single feature. While there's libraries to implement this, implementing it by hand is a good learning exercise.

In the end your ICE plot will look roughly like this:
![](https://christophm.github.io/interpretable-ml-book/images/ice-cervical-centered-1.jpeg)

Here's how to do it:
1. First some decisions. You need to pick: A feature to analyse, a sensible range of this feature, and a value for the feature where you want to center it (lowest value?).
2. For each datapoint in the test data, vary the feature that you're interested in over the range, and have your model make a prediction. This will result in many predictions for each datapoint.
3. Align the predictions by substracting the mean prediction at the feature value that you want to center (e.g. at age=14)
4. Plot a line for each datapoint with `plt.plot()`. Make sure you set the same color and some transparency (with `alpha=`).
5. Complete your plot by adding axes labels and a title.

Tip: As a sanity check, you can already try to plot the predicitons after 2. This will show you whether things are roughly working. This is also helpful, but slightly harder to interpret.

You'll notice that the density plot at the bottom, and the average yellow line are not there. If you want, you can try to add those as well. The yellow line is just the mean at each timepoint.

The density plot at the bottom is called a "raster plot", and is typically used to visualise neural spikes, either in Neuroscience on in Spiking Neural Networks. In matplotlib you can make this using `eventplot`: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.eventplot.html

Here, each "spike" is an observation of that value for your feature of interest. In the example image, it represents the age observed.


## Partial Dependence Plot with SKLearn

Of course you do not always need to implement these things by hand. SKLearn provides ready-made and easy to use methods that you can use to generate plots.

With PDP, we do not get a plot for each individual sample, but an average plot for the whole dataset.

You can use the `Partial Depdenence Display` class: https://scikit-learn.org/stable/modules/generated/sklearn.inspection.PartialDependenceDisplay.html

Using the standard initialization can be cumbersome, so you should use the `from_estimator()` method.

Here's what you'll do:
- Use `PartialDependenceDisplay.from_estimator(...)` to create a PDP plot of the same feature you investigated before. If your features are in a Pandas Dataframe you can use the name of the feature, otherwise use the index.
- Compare the two plots. Which plot do you find more useful? Do they give you the same insights?
- PDPs can also be plotted with two interacting features at the same time. Select two features where you think the interaction might be interesting, and plot the PDP.
- Interpret the interaction. Do you see anything interesting?



## SHapley Additive exPlanations (SHAP)

The previous visualisations gave explanations for single features, or at most two, but to get a picture of the whole model we would like to get explanations for all the important features.

We do this with Shapley values from Game Theory. The Shapley value for a given feature measures the difference between a model with this feature and a hypothetic model that does not have this feature.

This way it not only represents the weight towards a prediction (as linear regression parameters would), but also the importance.

The SHAP library allows you to calculate SHAP values for arbitrary models, and provides many functions for creating visualisations.

To get SHAP values we use the `shap.Explainer()` class. This explainer works on any ML model, though it is computationally expensive. It requires a `callable`, e.g. `model.predict_proba`, and the training data.

Then you can have it generate explanations by calling it with some test samples. Warning: SHAP is computationally expensive, so using your whole test dataset might take a while. Consider using only a subset of your test set. Also make sure you only need to call this cell once, so any extra analysis should be in another code cell. This will save on compute (and time).

After this you can play around with visualising your SHAP values.

Here's what you'll do:
1. Intiatilize explainer with `my_explainer = shap.Explainer(my_model.predict_proba, my_train_data)`
2. Call explainer with your test data `values = my_explainer(...)`
3. Create a bar plot using `shap.plots.bar`.
4. Create a beeswarm plot using `shap.plots.beeswarm`. Compare the results to the bar plot and take time to parse what is shown here. What does the visualisation mean? What does each dot's location and color represent? Do you think this is a better visualisation than the bar?
5. Create a waterfall plot using `shap.plots.waterfall`. This only works for a single sample at a time. Try a few different samples. Can you understand what is visualised? This would give an explanation for a single sample, which would be helpful in medical diagnosis. Do you think this is easy / helpful to interpret?

## Reflection

Look back at your first cell. There you built a model. The only thing you knew about it was it's test accuracy. You now have a better understanding of your model.

- Would you consider this a necessary step before putting your model into practice?
- Do you think your analysis is now sufficiently complete? Should you explore more things before this model can be put into production?
- Based on the SHAP values, do you know want to plot ICE/PDP for a different feature?
- In practice, you will be the AI expert, and you need to interpret these plots. What does this extra analysis tell you about the safety of your model? Do you think it behaves reasonably, in a way that we could use this to do medical diagnosis? What would make a model trustworthy?

If you're done with Part 1 - great job. You can take a breath, grab a coffee and get started on Part 2. When everyone has completed Part 1 we'll interrupt you and have a class discussion about Part 1.

# Part 2 - GradCAM

In this part we'll look at GradCAM as an Explainable AI method for Neural Networks. Ideally you would do this on tasks where it is important, such as medical imaging. However, such tasks are not trivial and making a good model for those tasks is not easy.

Instead, we'll use the MNIST handwritten digits dataset and a very simple Convolutional Neural Network. This is quick to train and do inference on, so allows us to learn quickly without waiting for models to converge.

I will provide you with the initial code to train the model. It comes directly from a Keras tutorial https://keras.io/examples/vision/mnist_convnet/.


Training the model will take about 2 minutes. We train for only 1 epochs, which is probably not enough for a good model, but it is good enough for now.

In [ ]:
import numpy as np
import keras
from keras import layers

# Model / data parameters
num_classes = 10
input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
# This is using Keras functional API. We connect each layer to the previous.
# This creates a string of layers following eachother.
# Keras will then learn which layers connect how.

inp = keras.Input(shape=input_shape)
x = layers.Conv2D(32, kernel_size=(3, 3), activation="relu")(inp)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
conv_activation = layers.Conv2D(64, kernel_size=(3, 3), activation="relu")(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(conv_activation)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
logit_activation = layers.Dense(num_classes)(x)
output = layers.Activation('softmax')(logit_activation)

model = keras.models.Model(
    inp, output # We only need to define where to start and where to end
)

model.summary()

In [ ]:
batch_size = 128
epochs = 1 # This is very few epochs. It won't be the best model, but it will work.

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

# Test accuracy should be >90%. Otherwise, something went wrong

## Implementing GradCAM by hand
Playing with hidden layers, gradients and backpropagation can be nice to reason about in theory, but implementing it in practice can be a hurdle.

We will go over implementing GradCAM by hand. The notebook will walk you through what to do roughly, but some debugging and thinking will be required. The goal is that after this, you can have some grasp on how to implement your own ideas when manipulating hidden layers.

At the same time, you will become more deeply familiar with how GradCAM works.

I'll first outline the setup, and then help you through the specific steps.

1. Create an alternative model where the outputs are the convolutional layer we're interested in, and the logits (output before softmax).
2. Perform a forward pass over this new model with a single sample, while recording the intermediate values so we can do backpropagation.
3. Determine which class is actually predicted (argmax of the logits)
4. Ask Keras to backpropagate from the logit of the predicted class, back to the convolutional layer we're intersted in.
5. For each filter in the convolutional layer, calculate the average of the gradient. In a sense, this uses the gradients to determine how important each filter was.
6. Multiply the activation of each filter in the convolutional layer with its corresponding average gradient. This gives us the activation of each filter, weighted by how it contributed to this classification
7. Plot the result as an 11x11 image.

Below I will outline how it will work in detail. I recommend implementing this step-by-step. After each step, check if things work. Print the relevant tensors that you calculated, or just their shapes. See if everything makes sense. It is normal that this is challenging. Take your time, print things to figure out what is happening, and ask Google, classmates and me for advice.


This is in more detail how you will implement GradCAM. Each enumeration belongs to a number above.
1. Define a new "model" for GradCAM, that starts at the inputs, and ends both at the last convolutional layer's activation (where we want to see activation) and the logits (activation before the softmax). This will look something like:
```
 keras.models.Model(inputs=input_layer, outputs=[convolutional_activation, logits_activation])
 ```

 Keras will then "know" which layers connect from input to your convolutional layer, and which layers connect from input to the logits_layer. I already made the activations we need accessible in the cell above.

2. While recording the gradients, make a forward pass. We can make a forward pass by calling the model with an input. Just write `model(your_input)`. Do not use `model.predict`. Your model will give two outputs, one for the convolutional activations, the other for the logits activations. We can make sure we record the gradients by doing this inside a

```
with tf.GradientTape() as tape:
    do_some_things()
```
  For every forward pass that happens inside the `with`, `tape` will keep track of the variables so we can do backpropagation later.

3. Determine which class is actually predicted (argmax of the logits). You'll need to make a prediction with your new model. It will give back two tensors. The first is the activation of the convolutional layer, the second is the logit activation. You can then find the predicted class with `tf.argmax`. Then select the index in the logits that we're interested in.

4. Ask Keras to backpropagate from the logit of the predicted class, back to the convolutional layer. You can do this by leaving the `with` and using `tape.gradient(end, start)`. This will backpropgate from end to start, and give back the gradients for our conv. layer. Have a look at their shape as well.

5. Average the gradients along the correct axes, to get one gradient for each filter. You can use `tf.reduce_mean()` for this.

6. Multiplty the activation of each filter with its average gradient. You might need to play around with shapes, then use `tf.matmul()` to do the multiplication.

7. You should now have an 11x11 tensor with your results. Use `plt.imshow()` to visualise it. Set `cmap='binary'` to make it look a little nicer.


It is normal that this task is more challenging. Take your time, try to figure out what's going on when you add new things by printing variables and shapes, and ask for help!

Try this with a couple of images separately. Do the patterns make sense?


## Bonus work

Too much time, and not enough exercises?

You can also look at LIME as a method for XAI. However, this comes with some extra hurdles as LIME's image explainer is only implemented for RGB images.

You'd need to get a pre-trained model for RGB images. You can follow the notebook they give, and see if you can get it to run here.

https://github.com/marcotcr/lime/blob/master/doc/notebooks/Tutorial%20-%20Image%20Classification%20Keras.ipynb

In their example they use Inception, which can predict many different classes. Find some pictures, see what predictions Inception gives, and what explanations LIME gives. Are they easy to interpet? Do you think the model is picking up on the right things?